
# Week 4 — FastAPI Deployment & Streamlit Dashboard
## AI-Driven Citizen Grievance & Sentiment Analysis System

**Objective:** Take all trained models from Weeks 1–3 and expose them as a production-ready REST API using **FastAPI**, then build an interactive **Streamlit dashboard** for civic officials to triage complaints in real time.

**What we build this week:**
- `app/main.py` — FastAPI server with 7 endpoints
- `streamlit_app/dashboard.py` — 4-page Streamlit civic dashboard
- `tests/test_api.py` — 20+ pytest unit & integration tests
- `docker-compose.yml` — one-command deployment of both services
- End-to-end API demo with live `requests` calls
- Performance benchmarking (throughput, latency)

---
**Roadmap:**
1. Install & Setup
2. Project Structure & Architecture
3. FastAPI Application Walkthrough
4. Start the API Server
5. End-to-End API Tests (live demo)
6. Batch Processing Demo
7. Performance Benchmarking
8. Run pytest Test Suite
9. Streamlit Dashboard Walkthrough
10. Docker Deployment
11. Full System Summary & KPIs

---
## Step 1: Install Dependencies

---
## Step 2: Project Structure & Architecture

```
week4/
├── app/
│   └── main.py                  ← FastAPI application (7 endpoints)
├── streamlit_app/
│   └── dashboard.py             ← Streamlit civic dashboard (4 pages)
├── tests/
│   └── test_api.py              ← 20+ pytest tests
├── models/                      ← Drop your .pkl and DistilBERT folder here
│   ├── best_department_classifier.pkl
│   ├── label_encoder.pkl
│   ├── urgency_classifier.pkl
│   ├── urgency_label_encoder.pkl
│   └── distilbert_urgency_final/
├── Dockerfile.api
├── Dockerfile.streamlit
├── docker-compose.yml
└── requirements.txt
```

### Architecture Diagram
```
Citizen / Gov Portal
        │
        ▼  HTTP POST /analyze
┌─────────────────────┐
│    FastAPI Server   │  ← uvicorn, port 8000
│  ┌───────────────┐  │
│  │  /analyze     │  │
│  │  /batch-analyze│  │
│  │  /stats       │  │
│  └───────┬───────┘  │
│          │           │
│  ┌───────▼───────┐  │
│  │  Model Layer  │  │
│  │ Dept: SVM     │  │
│  │ Urgency: BERT │  │
│  │ Fallback: Rules│  │
│  └───────────────┘  │
└──────────┬──────────┘
           │ JSON response
           ▼
┌─────────────────────┐
│  Streamlit Dashboard│  ← port 8501
│  (Civic Officials)  │
└─────────────────────┘
```

---
## Step 3: FastAPI Application Walkthrough

The API (`app/main.py`) contains 7 endpoints:

| Method | Endpoint | Purpose |
|---|---|---|
| GET | `/` | Root health check |
| GET | `/health` | Detailed model-load status |
| GET | `/departments` | All department labels |
| GET | `/urgency-levels` | Urgency levels + colors + descriptions |
| POST | `/analyze` | Single complaint → department + urgency |
| POST | `/batch-analyze` | Up to 100 complaints in one call |
| GET | `/stats` | Aggregate stats since server start |

### Key Design Decisions

**Lifespan model loading:** Models are loaded once at startup into a shared `models` dict using FastAPI's `lifespan` context manager — not reloaded on every request (which would be 100x slower).

**Graceful fallback chain:**
1. Try DistilBERT (most accurate)
2. Fall back to TF-IDF + LR (fast, still trained)
3. Fall back to keyword rules (always available, no model needed)

This means the API never crashes if a model file is missing — it degrades gracefully.

**Pydantic validation:** Input schemas validate minimum text length, reject empty strings, and cap batch size at 100 — before any inference code runs.

**In-memory stats:** Every request updates `stats_store` (dept counts, urgency counts, latency list) so the `/stats` endpoint always reflects live data.

---
## Step 4: Setup — Copy Models & Start the API Server

**Before running the cells below, copy your Week 2 & 3 model files:**

```bash
cp best_department_classifier.pkl  week4/models/
cp label_encoder.pkl               week4/models/
cp urgency_classifier.pkl          week4/models/
cp urgency_label_encoder.pkl       week4/models/
cp -r distilbert_urgency_final/    week4/models/
```

The API will work without these files too (using rule-based fallback) — so you can run the demo immediately.

In [ ]:
import os, subprocess, time, threading

# Create models directory
os.makedirs('week4/models', exist_ok=True)

# Copy model files if they exist in current directory
model_files = [
    'best_department_classifier.pkl',
    'label_encoder.pkl',
    'urgency_classifier.pkl',
    'urgency_label_encoder.pkl',
]
for f in model_files:
    if os.path.exists(f):
        import shutil
        shutil.copy(f, f'week4/models/{f}')
        print(f'  Copied {f}')
    else:
        print(f'  {f} not found — API will use rule-based fallback')

if os.path.isdir('distilbert_urgency_final'):
    import shutil
    shutil.copytree('distilbert_urgency_final', 'week4/models/distilbert_urgency_final', dirs_exist_ok=True)
    print('  Copied distilbert_urgency_final/')

print('\nModel directory contents:')
print(os.listdir('week4/models'))

In [ ]:
# Start FastAPI server in background thread
# (In production: uvicorn app.main:app --host 0.0.0.0 --port 8000 --workers 4)

server_process = None

def start_server():
    global server_process
    server_process = subprocess.Popen(
        ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
        cwd='week4',
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
time.sleep(4)   # wait for uvicorn to boot
print('FastAPI server started at http://localhost:8000')
print('Swagger UI: http://localhost:8000/docs')

---
## Step 5: End-to-End API Demo

In [ ]:
import requests, json

BASE = 'http://localhost:8000'

# ── Health Check ──────────────────────────────────────────────────────────────
r = requests.get(f'{BASE}/health')
health = r.json()
print('GET /health')
print(json.dumps(health, indent=2))

In [ ]:
# ── List Departments ──────────────────────────────────────────────────────────
r = requests.get(f'{BASE}/departments')
print('GET /departments')
print(json.dumps(r.json(), indent=2))

In [ ]:
# ── Single Complaint Analysis ─────────────────────────────────────────────────
test_complaints = [
    ('Critical', 'A live electrical wire has fallen near the school. Children are in DANGER!!!'),
    ('High',     'No water supply for the last 8 days. Filed 5 complaints. No action taken.'),
    ('Medium',   'Massive potholes on the main road. Several bike accidents this week.'),
    ('Low',      'The garbage bin at the corner was not emptied yesterday.'),
]

print('POST /analyze — Dual-output demo\n' + '='*60)
for expected_urgency, complaint in test_complaints:
    r = requests.post(f'{BASE}/analyze', json={'text': complaint, 'use_bert': False})
    d = r.json()
    match = '✅' if d['urgency'] == expected_urgency else '⚠️ '
    print(f'{match} Complaint : {complaint[:65]}...')
    print(f'   Department: {d["department"]}')
    print(f'   Urgency   : {d["urgency"]} (expected: {expected_urgency}) | confidence: {d["urgency_confidence"]}')
    print(f'   VADER     : {d["vader_score"]} | Model: {d["model_used"]}')
    print()

---
## Step 6: Batch Processing Demo

In [ ]:
import pandas as pd

batch_complaints = [
    'A live electrical wire has fallen near the primary school. Children are in DANGER!',
    'No water supply for the last 8 days. Filed 5 complaints but nothing done.',
    'Garbage bins near the market not collected this week.',
    'Massive potholes on main road causing vehicle damage.',
    'Sewage overflow near Block C spreading disease in the colony.',
    'Traffic signal at main junction not working. Accidents frequent.',
    'Street lights in our sector have been off for 3 nights.',
    'Drug peddling openly happening near the park. Need police.',
    'Low water pressure in taps for the past two days.',
    'Bus route 42 was cancelled without prior announcement.',
]

r = requests.post(f'{BASE}/batch-analyze',
                  json={'complaints': batch_complaints, 'use_bert': False})
data = r.json()

print(f'Processed {data["total"]} complaints in {data["summary"]["processing_time_seconds"]}s')
print(f'Critical: {data["summary"]["critical_count"]} | High: {data["summary"]["high_count"]}')
print('\nSummary by department:', json.dumps(data['summary']['by_department'], indent=2))
print('\nSummary by urgency   :', json.dumps(data['summary']['by_urgency'], indent=2))

In [ ]:
# ── Display results as a sorted priority queue ────────────────────────────────
urgency_order = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3}
results_sorted = sorted(data['results'], key=lambda r: urgency_order.get(r['urgency'], 99))

rows = []
for r in results_sorted:
    rows.append({
        'ID'         : r['complaint_id'],
        'Urgency'    : r['urgency'],
        'Department' : r['department'],
        'Confidence' : f"{r['urgency_confidence']*100:.0f}%",
        'VADER'      : r['vader_score'],
        'Complaint'  : r['original_text'][:55] + '...',
    })

df_results = pd.DataFrame(rows)
print('\n🏛️  CIVIC COMPLAINT PRIORITY QUEUE (sorted by urgency)')
print(df_results.to_string(index=False))

---
## Step 7: Performance Benchmarking

In [ ]:
import time, numpy as np, matplotlib.pyplot as plt

sample_complaints = [
    'No water in our area for 3 days.',
    'Pothole on main road damaged vehicle.',
    'Power outage for 5 hours.',
    'Garbage not collected this week.',
    'Live wire exposed near school. Dangerous!',
]

# ── Single-request latency ────────────────────────────────────────────────────
latencies = []
N = 50
for i in range(N):
    complaint = sample_complaints[i % len(sample_complaints)]
    t0 = time.time()
    requests.post(f'{BASE}/analyze', json={'text': complaint, 'use_bert': False})
    latencies.append((time.time() - t0) * 1000)

print(f'Single-request latency over {N} calls:')
print(f'  Mean : {np.mean(latencies):.1f} ms')
print(f'  Median: {np.median(latencies):.1f} ms')
print(f'  P95  : {np.percentile(latencies, 95):.1f} ms')
print(f'  P99  : {np.percentile(latencies, 99):.1f} ms')
print(f'  Max  : {max(latencies):.1f} ms')

# ── Batch throughput ──────────────────────────────────────────────────────────
batch_sizes = [5, 10, 20, 50, 100]
throughputs = []
for bs in batch_sizes:
    complaints = (sample_complaints * 20)[:bs]
    t0 = time.time()
    requests.post(f'{BASE}/batch-analyze', json={'complaints': complaints, 'use_bert': False})
    elapsed = time.time() - t0
    throughputs.append(bs / elapsed)
    print(f'  Batch {bs:3d}: {elapsed:.3f}s → {bs/elapsed:.1f} complaints/sec')

# ── Plots ─────────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Latency distribution
axes[0].hist(latencies, bins=20, color='#4e9af1', edgecolor='white', linewidth=0.8)
axes[0].axvline(np.mean(latencies), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {np.mean(latencies):.0f}ms')
axes[0].axvline(np.percentile(latencies, 95), color='orange', linestyle='--', linewidth=1.5, label=f'P95: {np.percentile(latencies,95):.0f}ms')
axes[0].set_xlabel('Latency (ms)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Single Request Latency Distribution', fontsize=12, fontweight='bold')
axes[0].legend()

# Throughput vs batch size
axes[1].plot(batch_sizes, throughputs, 'o-', color='#2ecc71', linewidth=2, markersize=7)
for x, y in zip(batch_sizes, throughputs):
    axes[1].annotate(f'{y:.0f}', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)
axes[1].set_xlabel('Batch Size (complaints)', fontsize=11)
axes[1].set_ylabel('Throughput (complaints/sec)', fontsize=11)
axes[1].set_title('Batch Throughput vs Batch Size', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('week4_performance_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: week4_performance_benchmark.png')

---
## Step 8: Run pytest Test Suite

In [ ]:
# Run all tests (TestClient doesn't need the server running — it spins up an internal test client)
!cd week4 && pytest tests/test_api.py -v --tb=short 2>&1

---
## Step 9: Streamlit Dashboard Walkthrough

The dashboard (`streamlit_app/dashboard.py`) has **4 pages**:

### Page 1 — Single Complaint Analyzer
- 4 quick-demo buttons (Critical / High / Medium / Low) pre-fill the text area
- Free-text complaint input
- Results: 4 metric cards (Department, Urgency, Confidence, VADER)
- Color-coded result box (red=Critical, orange=High, yellow=Medium, green=Low)
- Plotly gauge chart showing urgency confidence

### Page 2 — Batch Analysis
- Tab 1: Paste complaints (one per line)
- Tab 2: Upload a CSV with a complaint column
- Results: KPI row + department bar chart + urgency donut chart
- Full complaint queue sorted by urgency (Critical first)
- Download results as CSV

### Page 3 — Live Stats
- Pulls from `GET /stats` in real time
- Total processed, avg/P95 response time
- Department horizontal bar chart
- Urgency donut chart
- Manual refresh button

### Page 4 — API Info
- Full endpoint reference table
- Copy-paste curl examples
- Link to Swagger UI / ReDoc

### Sidebar
- API URL input (configurable — useful when deploying to a server)
- Live connection status + model-load indicator per model
- DistilBERT toggle

In [ ]:
# Start Streamlit dashboard in background (separate port)
dashboard_proc = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app/dashboard.py',
     '--server.port=8501', '--server.headless=true'],
    cwd='week4',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(5)
print('Streamlit dashboard started.')
print('Open in browser: http://localhost:8501')

---
## Step 10: Docker Deployment

Deploy both services with a single command:

```bash
cd week4

# Build and start both containers
docker-compose up --build

# Or in detached mode
docker-compose up --build -d

# Check running containers
docker-compose ps

# View API logs
docker-compose logs api

# Stop everything
docker-compose down
```

After startup:
- **FastAPI:**  http://localhost:8000
- **Swagger:**  http://localhost:8000/docs
- **Dashboard:** http://localhost:8501

### Production scaling with uvicorn workers:
```bash
uvicorn app.main:app --host 0.0.0.0 --port 8000 --workers 4
```
`--workers 4` spawns 4 parallel processes, each handling requests independently.

In [ ]:
# Verify Docker files exist
for f in ['week4/Dockerfile.api', 'week4/Dockerfile.streamlit', 'week4/docker-compose.yml']:
    exists = os.path.exists(f)
    print(f'{'✅' if exists else '❌'} {f}')

---
## Step 11: Full System Summary & KPIs

### Four-Week Engineering Summary

| Week | Focus | Key Deliverables |
|---|---|---|
| **1** | Data & EDA | Text preprocessing pipeline, word clouds, n-gram analysis, `processed_grievances.csv` |
| **2** | Classification | TF-IDF vectorization, SVM dept. classifier, cross-validation, `best_department_classifier.pkl` |
| **3** | Sentiment | VADER, TextBlob, DistilBERT fine-tuning, dual-output prediction, `urgency_classifier.pkl` |
| **4** | Deployment | FastAPI REST API, Streamlit dashboard, pytest suite, Docker compose |

### Target KPIs

| KPI | Target | How Measured |
|---|---|---|
| Department Classification Accuracy | ≥ 88% | Macro F1 on held-out test set (Week 2) |
| Urgency Macro F1 (minority class) | ≥ 80% | Macro F1 with `class_weight='balanced'` (Week 3) |
| API Single-request Latency (P95) | ≤ 200ms | Benchmarked in Week 4 (Step 7) |
| Batch throughput | ≥ 50 complaints/sec | Benchmarked in Week 4 (Step 7) |
| Test suite pass rate | 100% | 20+ pytest tests (Step 8) |

In [ ]:
# Final stats check
stats = requests.get(f'{BASE}/stats').json()
print('Final session stats:')
print(f'  Total complaints processed : {stats["total_processed"]}')
print(f'  By department              : {stats["by_department"]}')
print(f'  By urgency                 : {stats["by_urgency"]}')
print(f'  Avg response time          : {stats["avg_response_ms"]} ms')
print(f'  P95 response time          : {stats["p95_response_ms"]} ms')
print('\nProject complete!')

---
## Week 4 Deliverables Summary

| Deliverable | File | Status |
|---|---|---|
| FastAPI REST API | `app/main.py` | Done |
| 7 API endpoints (health, analyze, batch, stats...) | `app/main.py` | Done |
| Pydantic request/response schemas | `app/main.py` | Done |
| Graceful 3-tier model fallback | `app/main.py` | Done |
| Streamlit 4-page civic dashboard | `streamlit_app/dashboard.py` | Done |
| Single complaint analyzer page | `streamlit_app/dashboard.py` | Done |
| Batch analysis + CSV upload page | `streamlit_app/dashboard.py` | Done |
| Live stats analytics page | `streamlit_app/dashboard.py` | Done |
| 20+ pytest tests | `tests/test_api.py` | Done |
| Performance benchmark | Step 7 in notebook | Done |
| Latency / throughput charts | `week4_performance_benchmark.png` | Done |
| Docker Compose deployment | `docker-compose.yml` | Done |
| Dockerfiles (API + dashboard) | `Dockerfile.api/streamlit` | Done |
| Requirements file | `requirements.txt` | Done |